# Week 4: Model Evaluation, Validation, and Testing - Demo

This script mirrors the sections of the written report on an imbalanced
fraud-style classification problem (about 3% positives):

1. Data splitting (stratified train / validation / test)
2. Baselines and the accuracy paradox
3. Metrics: confusion matrix, precision, recall, F1, ROC-AUC, PR-AUC, MCC, Brier
4. Validation: stratified K-fold and nested cross-validation
5. Threshold selection on the validation set
6. Final test evaluation with bootstrap confidence intervals
7. Error analysis (slice-based)
8. Testing strategy: edge cases, noise robustness, invariance

Run: python src/evaluation_demo.py   (figures are saved to ./figures)

In [ ]:
import os
import warnings

import matplotlib
matplotlib.use("Agg")  # works without a display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, matthews_corrcoef, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

warnings.filterwarnings("ignore")
SEED = 42
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # running inside a notebook
    _HERE = os.path.join(os.getcwd(), "src")
FIG_DIR = os.path.join(_HERE, "..", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)


def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"   saved figure -> figures/{name}")

## 1. Data and stratified splitting
Stratification keeps the 3% positive rate identical in every split.
Fitting happens only on the training set; the test set stays locked until Step 6.

In [ ]:
X, y = make_classification(
    n_samples=20000, n_features=12, n_informative=6, n_redundant=2,
    weights=[0.97, 0.03], class_sep=1.0, flip_y=0.01, random_state=SEED,
)
feature_names = [f"f{i}" for i in range(X.shape[1])]
X = pd.DataFrame(X, columns=feature_names)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

print("== 1. Splits ==")
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"   {name:5s}: n={len(yy):5d}  positive rate={yy.mean():.3%}")

## 2. Baseline and the accuracy paradox
A model that always predicts "not fraud" looks excellent on accuracy and
is useless. This is why the primary metric must match the error costs.

In [ ]:
print("\n== 2. Baseline (always predict majority class) ==")
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred = dummy.predict(X_val)
print(f"   accuracy = {accuracy_score(y_val, pred):.3f}")
print(f"   recall   = {recall_score(y_val, pred):.3f}   <- catches nothing")
print(f"   F1       = {f1_score(y_val, pred, zero_division=0):.3f}")

## 3. Candidate models (preprocessing inside the pipeline = no leakage)

In [ ]:
def make_logreg():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
    ])


def make_rf():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=200, class_weight="balanced_subsample",
            min_samples_leaf=5, n_jobs=-1, random_state=SEED)),
    ])


models = {"LogisticRegression": make_logreg(), "RandomForest": make_rf()}

## 4a. Stratified 5-fold cross-validation
Report mean and standard deviation, not a single number. The standard
deviation tells you how stable the estimate is.

In [ ]:
print("\n== 4a. Stratified 5-fold CV on the training set ==")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"precision": "precision", "recall": "recall", "f1": "f1",
           "roc_auc": "roc_auc", "pr_auc": "average_precision"}
cv_rows = []
for name, model in models.items():
    res = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring)
    row = {"model": name}
    for m in scoring:
        vals = res[f"test_{m}"]
        row[m] = f"{vals.mean():.3f} +/- {vals.std():.3f}"
    cv_rows.append(row)
cv_table = pd.DataFrame(cv_rows).set_index("model")
print(cv_table.to_string())
cv_table.to_csv(os.path.join(FIG_DIR, "cv_results.csv"))

## 4b. Nested cross-validation (honest estimate when tuning)
Inner loop tunes hyperparameters; outer loop scores the whole procedure.
Reusing the same folds for tuning and reporting gives optimistic results.

In [ ]:
print("\n== 4b. Nested CV (RandomForest, tuning min_samples_leaf) ==")
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED + 1)
param_grid = {"clf__min_samples_leaf": [2, 5, 10]}
outer_scores = []
for fold, (tr, te) in enumerate(outer.split(X_train, y_train), 1):
    search = GridSearchCV(make_rf(), param_grid, cv=inner,
                          scoring="average_precision", n_jobs=1)
    search.fit(X_train.iloc[tr], y_train[tr])
    proba = search.predict_proba(X_train.iloc[te])[:, 1]
    score = average_precision_score(y_train[te], proba)
    outer_scores.append(score)
    print(f"   fold {fold}: best={search.best_params_}  outer PR-AUC={score:.3f}")
print(f"   nested PR-AUC = {np.mean(outer_scores):.3f} +/- {np.std(outer_scores):.3f}")

## 5. Model selection and threshold choice on the validation set
Business rule used here: reach recall >= 0.80, then maximise precision.

In [ ]:
print("\n== 5. Validation-set comparison and threshold selection ==")
for m in models.values():
    m.fit(X_train, y_train)

val_summary = {}
for name, m in models.items():
    p = m.predict_proba(X_val)[:, 1]
    val_summary[name] = {"roc_auc": roc_auc_score(y_val, p),
                         "pr_auc": average_precision_score(y_val, p),
                         "brier": brier_score_loss(y_val, p)}
print(pd.DataFrame(val_summary).T.round(3).to_string())

best_name = max(val_summary, key=lambda k: val_summary[k]["pr_auc"])
best_model = models[best_name]
print(f"   selected model (highest validation PR-AUC): {best_name}")

TARGET_RECALL = 0.80
p_val = best_model.predict_proba(X_val)[:, 1]
prec, rec, thr = precision_recall_curve(y_val, p_val)
ok = np.where(rec[:-1] >= TARGET_RECALL)[0]
best_i = ok[np.argmax(prec[:-1][ok])] if len(ok) else np.argmax(prec[:-1] * rec[:-1])
THRESHOLD = float(thr[best_i])
print(f"   chosen threshold = {THRESHOLD:.3f}  "
      f"(val precision={prec[best_i]:.3f}, recall={rec[best_i]:.3f})")

plt.figure(figsize=(6, 4.5))
plt.plot(rec, prec, label=f"{best_name} (PR-AUC={val_summary[best_name]['pr_auc']:.2f})")
plt.axhline(y_val.mean(), ls="--", c="grey", label="no-skill baseline")
plt.scatter([rec[best_i]], [prec[best_i]], c="red", zorder=3, label="chosen threshold")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall curve (validation)")
plt.legend(); savefig("pr_curve_validation.png")

## 6. Final test evaluation (run once) with bootstrap confidence intervals

In [ ]:
print("\n== 6. Final test-set evaluation (touched once) ==")
p_test = best_model.predict_proba(X_test)[:, 1]
y_hat = (p_test >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_hat).ravel()
print(f"   confusion matrix: TP={tp} FP={fp} FN={fn} TN={tn}")


def bootstrap_ci(metric, y_true, y_pred, n=1000, alpha=0.05):
    idx = np.arange(len(y_true)); vals = []
    for _ in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        if y_true[s].sum() == 0:
            continue
        vals.append(metric(y_true[s], y_pred[s]))
    return np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])


metrics = {
    "precision": (precision_score, y_hat),
    "recall": (recall_score, y_hat),
    "f1": (f1_score, y_hat),
    "mcc": (matthews_corrcoef, y_hat),
    "roc_auc": (roc_auc_score, p_test),
    "pr_auc": (average_precision_score, p_test),
}
for name, (fn_, pred_) in metrics.items():
    point = fn_(y_test, pred_)
    lo, hi = bootstrap_ci(fn_, y_test, pred_)
    print(f"   {name:9s} {point:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")
print(f"   brier     {brier_score_loss(y_test, p_test):.4f}")

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
fpr, tpr, _ = roc_curve(y_test, p_test)
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "--", c="grey")
ax[0].set(title=f"ROC (AUC={roc_auc_score(y_test, p_test):.3f})",
          xlabel="False positive rate", ylabel="True positive rate")
pr_, rc_, _ = precision_recall_curve(y_test, p_test)
ax[1].plot(rc_, pr_); ax[1].axhline(y_test.mean(), ls="--", c="grey")
ax[1].set(title=f"PR (AUC={average_precision_score(y_test, p_test):.3f})",
          xlabel="Recall", ylabel="Precision")
frac_pos, mean_pred = calibration_curve(y_test, p_test, n_bins=8, strategy="quantile")
ax[2].plot(mean_pred, frac_pos, "o-"); ax[2].plot([0, 1], [0, 1], "--", c="grey")
ax[2].set(title="Calibration (reliability diagram)",
          xlabel="Mean predicted probability", ylabel="Observed fraction positive")
savefig("test_curves.png")

cm = confusion_matrix(y_test, y_hat)
plt.figure(figsize=(4.2, 3.8))
plt.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center",
             color="white" if v > cm.max() / 2 else "black")
plt.xticks([0, 1], ["Pred 0", "Pred 1"]); plt.yticks([0, 1], ["Actual 0", "Actual 1"])
plt.title("Confusion matrix (test)"); savefig("confusion_matrix.png")

## 7. Error analysis: slice-based evaluation
A good overall score can hide a weak subgroup. Here we slice by quartiles of
one feature and compare recall, then list the highest-confidence mistakes.

In [ ]:
print("\n== 7. Error analysis ==")
test_df = X_test.copy()
test_df["y"] = y_test; test_df["p"] = p_test; test_df["pred"] = y_hat
test_df["slice"] = pd.qcut(test_df["f0"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
rows = []
for s, g in test_df.groupby("slice", observed=True):
    rows.append({"slice (f0 quartile)": s, "n": len(g), "positives": int(g.y.sum()),
                 "recall": recall_score(g.y, g.pred, zero_division=0),
                 "precision": precision_score(g.y, g.pred, zero_division=0)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

fn_df = test_df[(test_df.y == 1) & (test_df.pred == 0)].sort_values("p")
print(f"   false negatives: {len(fn_df)}; lowest-scored missed positives (inspect manually):")
print(fn_df[["f0", "f1", "f2", "p"]].head(5).round(3).to_string())

## 8. Testing strategy: edge cases, robustness, invariance

In [ ]:
print("\n== 8. Edge-case and robustness tests ==")
results = []


def check(name, passed, detail=""):
    results.append(passed)
    print(f"   [{'PASS' if passed else 'FAIL'}] {name} {detail}")


# 8.1 Missing values: the pipeline must not crash and must return valid probabilities
X_nan = X_test.copy().iloc[:500]
X_nan.iloc[::7, 0] = np.nan
p_nan = best_model.predict_proba(X_nan)[:, 1]
check("missing values handled", np.isfinite(p_nan).all() and ((p_nan >= 0) & (p_nan <= 1)).all())

# 8.2 Extreme values: output stays a valid probability
X_ext = X_test.copy().iloc[:500] * 1000
p_ext = best_model.predict_proba(X_ext)[:, 1]
check("extreme magnitude inputs give valid probabilities",
      np.isfinite(p_ext).all() and ((p_ext >= 0) & (p_ext <= 1)).all())

# 8.3 Noise robustness: small perturbation should barely change predictions
noise = rng.normal(0, 0.05, X_test.shape)
p_noisy = best_model.predict_proba(X_test + noise)[:, 1]
flip_rate = np.mean((p_noisy >= THRESHOLD) != (p_test >= THRESHOLD))
check("small noise flips few decisions", flip_rate < 0.05, f"(flip rate={flip_rate:.2%})")

# 8.4 Determinism: same input, same output
p_again = best_model.predict_proba(X_test)[:, 1]
check("deterministic predictions", np.allclose(p_test, p_again))

# 8.5 Invariance: row order must not change a row's prediction
perm = rng.permutation(len(X_test))
p_perm = best_model.predict_proba(X_test.iloc[perm])[:, 1]
check("row-order invariance", np.allclose(p_test[perm], p_perm))

# 8.6 Directional expectation: pushing a positively-weighted feature up should not
#     lower the score on average (checked on the logistic model's own coefficient)
lr = models["LogisticRegression"]
coefs = lr.named_steps["clf"].coef_[0]
j = int(np.argmax(np.abs(coefs)))
X_up = X_test.copy(); X_up.iloc[:, j] += 0.5 * np.sign(coefs[j])
delta = (lr.predict_proba(X_up)[:, 1] - lr.predict_proba(X_test)[:, 1]).mean()
check(f"directional test on feature f{j}", delta > 0, f"(mean score change={delta:+.4f})")

# 8.7 Regression guard: new model must not be worse than the baseline by a margin
check("beats no-skill PR-AUC baseline",
      average_precision_score(y_test, p_test) > 2 * y_test.mean())

print(f"\n   {sum(results)}/{len(results)} checks passed")
print("\nDone. Figures and cv_results.csv are in ./figures")